In [ ]:
%pip install -q -r requirements.txt

# Adaptive Consistency Layer - Experiment Runner

Runs the whole evaluation from one notebook: environment and latency-injection check, tests, the **benchmark sweep** (3 consistency configurations x 9 traffic presets x injected inter-node latency levels x repeats), the **staleness sweep**, and tables and charts.

### What is being tested

The adaptive layer picks FAST, MODERATE or SAFE settings from the **write share** of recent traffic (writes / all operations; `adaptive_layer/decision.py`): under 15% writes is FAST, over 40% is SAFE, in between is MODERATE. The thresholds were fixed before this evaluation and are not adjusted to its results. An earlier evaluation used an absolute writes-per-second rule that put every preset in SAFE, so adaptive behaved like Static Safe; that is why this rule was replaced.

Each adaptive run also records **which settings it actually used and when** (`<config>_<preset>_switches.csv`), shown in Section 7.4, so claims about adaptive's choices are observed, not inferred. Every run also mixes in **staleness probes** (about 2% of iterations: a write followed immediately by a read of that document); a probe whose read misses is a stale read. That measures staleness under the real traffic, tallied by the settings in force (Section 7.7).

### Why latency is injected

All three MongoDB nodes run on one machine, so a `w: "majority"` write costs almost nothing and the consistency trade-off is invisible. `tc netem` (inside each container) delays **only node-to-node traffic** (replication, majority acknowledgement, secondary lag), not the client's own connection. `delay` is **one-way** per node, so the node-to-node round trip is about 2x.

| one-way delay | scenario | round trip it models | source |
|---|---|---|---|
| 0 ms | baseline (single host) | none | - |
| 5 ms | neighbouring regions | ~8-9 ms | Azure P50, East US <-> East US 2 ([Microsoft Learn](https://learn.microsoft.com/en-us/azure/networking/azure-network-latency)) |
| 43 ms | transatlantic | ~85-89 ms | Azure P50, East US <-> West Europe (same page) |

(1 ms same-AZ and 33 ms cross-country remain available through `LATENCY_LEVELS_MS`; see `benchmark/netem.py`.) The same delay is applied to all three configurations at each level.

**Prerequisites:** Docker Desktop running; `docker compose up -d --build` from the repo root (the compose file builds an image that adds `tc`); replica set initiated (see `docs/replica_set_setup.md`); this notebook's kernel set to the project `.venv`.

**Runtime:** printed in Section 0 from your settings. Full-length runs with 3 repeats are on the order of a day; `DURATION_SCALE` shortens every run.

Results go to `results/write_share_eval/` (a separate folder, so earlier results are never overwritten).

## 0. Setup and configuration

In [ ]:
import sys, os, subprocess, time
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "adaptive_layer").exists():
    for parent in Path.cwd().parents:
        if (parent / "adaptive_layer").exists():
            REPO_ROOT = parent
            break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Repo root:", REPO_ROOT)
print("Python:   ", sys.executable)

In [ ]:
# ------------------------- experiment configuration -------------------------
QUICK = True                                   # True: short runs for a smoke test. False: full-length runs
REPEATS = 1 if QUICK else 3                    # repeats of the whole sweep (mean and spread are reported)
LATENCY_LEVELS_MS = [0, 5, 43]                 # one-way inter-node delay per node, in ms
PRESETS_TO_RUN = None                          # None = all 9 presets, or e.g. ["READ_HEAVY", "WRITE_HEAVY"]
DURATION_SCALE = 1.0                           # multiplies every run length (0.4 turns 300 s into 120 s)
STALENESS_TRIALS = 40 if QUICK else 200        # trials per staleness scenario
RUN_TESTS = True                               # run the pytest suite in section 2
# -----------------------------------------------------------------------------

from benchmark import netem, run_benchmark as rb
PRESETS = PRESETS_TO_RUN or rb.PRESETS
n_runs = REPEATS * len(LATENCY_LEVELS_MS) * len(rb.CONFIGS) * len(PRESETS)
bench_seconds = REPEATS * len(LATENCY_LEVELS_MS) * len(rb.CONFIGS) * sum(
    int(rb.params_for(p, QUICK, DURATION_SCALE)["duration"].rstrip("s")) + 5 for p in PRESETS)
print(f"Mode: {'QUICK' if QUICK else 'FULL'} | {REPEATS} repeat(s) x {len(LATENCY_LEVELS_MS)} latency levels "
      f"{LATENCY_LEVELS_MS} x {len(rb.CONFIGS)} configs x {len(PRESETS)} presets = {n_runs} benchmark runs")
print(f"Estimated benchmark time: about {bench_seconds / 3600:.1f} hours (run time only), plus the staleness sweep")

**Emergency reset:** if you interrupt a run mid-way, latency may still be injected. Run this cell to remove it (the sweeps below also clear it automatically when they finish or fail).

In [ ]:
netem.clear()
print("Injected latency cleared. Node RTT now:", round(netem.measure_rtt_ms("mongo1", "mongo2"), 2), "ms")

## 1. Environment check

In [ ]:
from pymongo import MongoClient

URI = "mongodb://mongo1:27017,mongo2:27018,mongo3:27019/?replicaSet=rs0"
_client = MongoClient(URI, serverSelectionTimeoutMS=5000)
try:
    status = _client.admin.command("replSetGetStatus")
    for m in status["members"]:
        print(f"{m['name']:<16} {m['stateStr']}")
    n_primary = sum(m["stateStr"] == "PRIMARY" for m in status["members"])
    n_secondary = sum(m["stateStr"] == "SECONDARY" for m in status["members"])
    assert n_primary == 1 and n_secondary == 2, "expected 1 PRIMARY + 2 SECONDARY"
    print("\nReplica set healthy.")
finally:
    _client.close()

## 2. Unit / integration tests

Runs `tests/` (monitor, decision engine, feedback loop, switch log, runner and analysis plumbing, and the live tests against the replica set) before trusting any numbers below.

In [ ]:
if RUN_TESTS:
    result = subprocess.run([sys.executable, "-m", "pytest", "tests/", "-q", "--color=no"],
                            cwd=REPO_ROOT, capture_output=True, text=True)
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-1500:])
    print("Exit code:", result.returncode)
else:
    print("Skipped (RUN_TESTS = False)")

## 3. Verify the latency injection

Applies each level, measures the real ping round trip between two nodes, then removes it. Expect the measured round trip to be a bit above 2x the one-way delay (plus ~1 ms overhead).

In [ ]:
import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 50)

rows = []
try:
    for d in LATENCY_LEVELS_MS:
        netem.apply_latency(d)
        rows.append({
            "one_way_delay_ms": d,
            "scenario": netem.LATENCY_LABELS.get(d, "custom"),
            "expected_rtt_ms": 2 * d,
            "measured_rtt_ms": round(netem.measure_rtt_ms("mongo1", "mongo2"), 1),
        })
finally:
    netem.clear()
pd.DataFrame(rows)

## 4. Benchmark sweep

For each repeat and latency level: inject the delay, run every `config x preset` through Locust (one subprocess each), write the stats CSVs (and, for adaptive, the switch log) to `results/write_share_eval/latency_<N>ms/rep<k>/`, then clear the delay. Output is long: one line per run.

In [ ]:
bench_runs = {}
for rep in range(1, REPEATS + 1):
    for lat in LATENCY_LEVELS_MS:
        print(f"\n##### repeat {rep}/{REPEATS}, {lat} ms one-way ({netem.LATENCY_LABELS.get(lat, 'custom')}) #####")
        bench_runs[(rep, lat)] = rb.run_suite(lat, QUICK, presets=PRESETS, rep=rep, duration_scale=DURATION_SCALE)

failed = [(k, c) for k, r in bench_runs.items() for c, v in r.items() if v is None]
print("\nAll runs produced results:", not failed)
if failed:
    print("Failed:", failed)

## 5. Staleness sweep

Write a unique marker, immediately read it back with the configuration's read settings, and count misses, at each latency level and repeat. Each trial also records which settings were in force, so the output says how many trials really ran under the settings each scenario is named for. `adaptive_cold` is deliberately paced, so it is **not** directly comparable with `static_fast`; `adaptive_hot` versus `static_safe` and `static_fast` is a fair comparison.

In [ ]:
from benchmark import staleness_test as st

stale_runs = {}
for rep in range(1, REPEATS + 1):
    for lat in LATENCY_LEVELS_MS:
        print(f"\n##### repeat {rep}/{REPEATS}, {lat} ms one-way ({netem.LATENCY_LABELS.get(lat, 'custom')}) #####")
        stale_runs[(rep, lat)] = st.run_suite(lat, STALENESS_TRIALS, rep)
print("\nDone.")

## 6. Load and aggregate

Reads the repeats under `results/write_share_eval/` for the levels and repeats chosen above, aggregates them (mean and standard deviation over repeats; staleness trials pooled), and writes the aggregate CSVs and all figures.

In [ ]:
import pandas as pd
from benchmark import analyze_results as ar
from IPython.display import Image, display

res = ar.run_analysis(rb.EVAL_DIR, verbose=False, levels=LATENCY_LEVELS_MS, max_rep=REPEATS)
bench_reps, bench_agg, stale_reps, stale_pooled = res.bench_reps, res.bench_agg, res.stale_reps, res.stale_pooled
tl, tl_agg = res.timelines, res.timelines_agg

ORDER = ["static_fast", "static_safe", "adaptive"]
LEVELS = sorted(bench_agg["latency_ms"].unique())
print(f"{len(bench_reps)} benchmark rows ({REPEATS} repeat(s)), {len(stale_reps)} staleness rows, "
      f"failed requests in total: {int(bench_reps['failures'].sum())}")


def mean_std_table(metric, digits=0):
    # one cell per (preset, latency, config): "mean +- std over repeats" (just the mean for a single repeat)
    d = bench_agg.copy()
    std = d[metric + "_std"]
    d["cell"] = [f"{m:.{digits}f}" if pd.isna(s) else f"{m:.{digits}f} ± {s:.{digits}f}" for m, s in zip(d[metric], std)]
    t = d.pivot_table(index="preset", columns=["latency_ms", "config"], values="cell", aggfunc="first")
    return t.reindex(columns=[(l, c) for l in LEVELS for c in ORDER])

## 7. Tables

### 7.1 Run overview

In [ ]:
bench_reps.groupby(["latency_ms", "rep"]).agg(runs=("config", "size"), requests=("requests", "sum"),
                                                failures=("failures", "sum"))

### 7.2 p95 latency (ms), mean ± std over repeats

In [ ]:
mean_std_table("p95_ms")

### 7.3 Where the latency goes: write and read separately (median, ms)

On workloads with a mix of reads and writes the aggregate median sits between two very different populations (fast reads, slow majority writes) and is unstable, so reads and writes are shown on their own.

In [ ]:
print("Write median (ms):")
display(mean_std_table("write_p50_ms"))
print("Read median (ms):")
display(mean_std_table("read_p50_ms"))

### 7.4 Settings the adaptive layer actually used

Share of each run (mean over repeats) spent in each setting, from the switch log, and the average number of switches. This is the evidence for whether adaptive used FAST and MODERATE or stayed in SAFE.

In [ ]:
tl_agg.rename(columns={"FAST_pct": "FAST %", "MODERATE_pct": "MODERATE %", "SAFE_pct": "SAFE %"}).round(1).sort_values(
    ["preset", "latency_ms"]).reset_index(drop=True)

### 7.5 Adaptive versus each static baseline (p95)

In [ ]:
p95 = bench_agg.pivot_table(index=["latency_ms", "preset"], columns="config", values="p95_ms")[ORDER]
cmp_ = pd.DataFrame({
    "adaptive_ms": p95["adaptive"],
    "vs_static_fast_%": ((p95["adaptive"] / p95["static_fast"] - 1) * 100).round(0),
    "vs_static_safe_%": ((p95["adaptive"] / p95["static_safe"] - 1) * 100).round(0),
    "best": p95.idxmin(axis=1),
})
cmp_

### 7.6 Read-after-write staleness

Pooled over repeats (stale reads / trials), and the share of trials that ran under the settings each scenario is named for.

In [ ]:
s = stale_pooled.copy()
s["stale"] = [f"{m}/{n} ({r:.1%})" for m, n, r in zip(s["misses"], s["trials"], s["stale_rate"])]
s["under intended settings"] = [f"{k / n:.0%}" for k, n in zip(s["in_expected_settings"], s["trials"])]
display(s.pivot_table(index="latency_ms", columns="config", values="stale", aggfunc="first")[
    ["static_fast", "static_safe", "adaptive_hot", "adaptive_cold"]])
print("Trials that ran under the intended settings:")
s.pivot_table(index="latency_ms", columns="config", values="under intended settings", aggfunc="first")[
    ["static_fast", "static_safe", "adaptive_hot", "adaptive_cold"]]

### 7.7 Staleness measured inside the load test

Stale probe reads / probes, pooled over repeats, for each preset. A probe is a write followed immediately by a read of that same document, issued by about 2% of iterations while the normal traffic runs, so this is staleness under real load (the isolated experiment in 7.6 runs on an otherwise idle cluster). Static Safe should read 0%.

In [ ]:
def probe_cells(df):
    d = df.copy()
    d["cell"] = [f"{s}/{n} ({p:.0f}%)" if n else "-" for s, n, p in zip(d["stale"], d["probes"], d["stale_pct"].fillna(0))]
    return d

if res.probes_pooled.empty:
    print("No probe data (probes were disabled or no run wrote a probe file).")
else:
    t = probe_cells(res.probes_pooled).pivot_table(index="preset", columns=["latency_ms", "config"], values="cell", aggfunc="first")
    display(t.reindex(columns=[(l, c) for l in LEVELS for c in ORDER]))

Adaptive's probes split by the settings in force when the read ran. This separates the stale rate while adaptive was in FAST (or MODERATE) from the rate while it was in SAFE, which is what shows whether its FAST decisions cost freshness.

In [ ]:
if not res.probes_by_settings.empty:
    ad = probe_cells(res.probes_by_settings[res.probes_by_settings["config"] == "adaptive"])
    display(ad.pivot_table(index=["preset", "latency_ms"], columns="settings", values="cell", aggfunc="first"))

## 8. Charts across latency levels

### 8.1 p50 latency vs injected latency (one panel per preset, error bars = std over repeats)

In [ ]:
display(Image(filename=str(rb.EVAL_DIR / "fig_sweep_p50.png")))

### 8.2 p95 latency vs injected latency

In [ ]:
display(Image(filename=str(rb.EVAL_DIR / "fig_sweep_p95.png")))

### 8.3 Staleness vs injected latency

In [ ]:
display(Image(filename=str(rb.EVAL_DIR / "fig_sweep_staleness.png")))

### 8.4 Settings the adaptive layer actually used

In [ ]:
display(Image(filename=str(rb.EVAL_DIR / "fig_settings_used.png")))

### 8.5 Staleness measured inside the load test

In [ ]:
display(Image(filename=str(rb.EVAL_DIR / "fig_probe_staleness.png")))

## 9. Charts per latency level

### 9.1 Latency percentiles (p50 / p95 / p99), grouped by preset

In [ ]:
for lat in LEVELS:
    print(f"--- {lat} ms one-way ({netem.LATENCY_LABELS.get(lat, 'custom')}) ---")
    display(Image(filename=str(rb.EVAL_DIR / f"latency_{lat}ms" / "fig_latency_percentiles.png")))

### 9.2 Throughput

In [ ]:
for lat in LEVELS:
    print(f"--- {lat} ms one-way ---")
    display(Image(filename=str(rb.EVAL_DIR / f"latency_{lat}ms" / "fig_throughput.png")))

### 9.3 Staleness by scenario

In [ ]:
for lat in LEVELS:
    print(f"--- {lat} ms one-way ---")
    display(Image(filename=str(rb.EVAL_DIR / f"latency_{lat}ms" / "fig_staleness.png")))

## 10. Summary by latency level

Averages across the presets that were run. Use it as a starting point for the write-up, then check the per-preset tables above: averages hide the presets where adaptive loses.

In [ ]:
rows = []
for lat, g in cmp_.groupby(level="latency_ms"):
    sp = stale_pooled[stale_pooled["latency_ms"] == lat].set_index("config")["stale_rate"] * 100
    tl_l = tl_agg[tl_agg["latency_ms"] == lat]
    rows.append({
        "one_way_delay_ms": lat,
        "adaptive_vs_fast_p95_%": round(g["vs_static_fast_%"].mean(), 0),
        "adaptive_vs_safe_p95_%": round(g["vs_static_safe_%"].mean(), 0),
        "presets_adaptive_best": int((g["best"] == "adaptive").sum()),
        "presets_total": len(g),
        "adaptive_time_FAST_%": round(tl_l["FAST_pct"].mean(), 0) if len(tl_l) else float("nan"),
        "adaptive_time_SAFE_%": round(tl_l["SAFE_pct"].mean(), 0) if len(tl_l) else float("nan"),
        "stale_static_fast_%": round(sp.get("static_fast", float("nan")), 1),
        "stale_static_safe_%": round(sp.get("static_safe", float("nan")), 1),
        "stale_adaptive_hot_%": round(sp.get("adaptive_hot", float("nan")), 1),
    })
pd.DataFrame(rows).set_index("one_way_delay_ms")

### How to read this

- **Adaptive is worth it** where it is close to the faster of the two baselines on latency *and* close to the safer one on staleness, **and** the settings table (7.4) shows it really used FAST or MODERATE for the traffic that calls for them.
- **Adaptive is not worth it** where Static Safe (or Static Fast) is simply better on both axes for a preset. If adaptive spends nearly all its time in SAFE, it is Static Safe with extra moving parts.
- `adaptive_hot` vs `static_safe` is a fair staleness comparison; `adaptive_cold` is paced and is not comparable to `static_fast`. Check the "ran under intended settings" table before reading its staleness rate.
- A standard deviation larger than the gap between two configurations means the gap is not established by these runs.
- Staleness here is the worst case (an immediate read after a write); many applications tolerate short staleness.
- The in-load probes (7.7) are a few percent of the traffic, so their counts are small in quick runs; read their rates only when the probe counts are large enough to mean something.